# Part C — Project description and toolbox

**Demonstration time:** about 45 minutes

This notebook is an instructor-led tour of the controls available for
the group project. It is a reference, not a worksheet and not a report
template. The two demonstrations show how an input map becomes a model
experiment and how to keep the mapped information visible in an
animation.

The project itself should still be a controlled experiment: start from
one baseline, vary one primary factor, and support the conclusion with a
quantitative diagnostic rather than an animation alone.


## 1. From a physical question to a numerical experiment

A useful project has four connected pieces:

1. a focused question and a prediction based on a mechanism;
2. a baseline configuration that runs successfully;
3. one or, if time permits, two controlled variations of one parameter;
4. a diagnostic such as speed, arrival time, amplitude, period, or
   reflection time.

The model is a fully wet, one-layer shallow-water model. A coastline is
a rigid wall; there is no run-up, inundation, breaking, or wetting and
drying. File-backed fields must already match the model grid.


## 2. Controls available for Part C

| Category | Examples of controls |
|---|---|
| Wind | stress amplitude, direction, duration, spatial map |
| Bottom | uniform depth, shelf depth/width, ridge, file map |
| Domain | `Lx`, `Ly`, aspect ratio, `Nx`, `Ny` |
| Initial state | amplitude, radius, position, circular or cross-basin shape |
| Rotation | constant Coriolis parameter `f` |
| Dissipation | Rayleigh damping timescale |

Domain size and resolution are separate controls. Changing `Lx` while
keeping `Nx` fixed also changes `dx`, so both must be reported. Rotation
matters only if the integration time is a meaningful fraction of the
inertial period $2\pi/|f|$ or the domain is comparable to the deformation
radius $\sqrt{gH}/|f|$.


## 3. Imports and reusable display helper

`animate_with_overlay(...)` uses the same `animate_eta` function as Parts
A and B. An optional drawing function adds persistent contours or arrows
before the GIF is saved. This is useful for keeping the field that causes
the response visible behind the evolving surface elevation $\eta$.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, animate_eta, backend_info, compute_dt_cfl, depth_on_u,
    load_bathymetry, make_grid, make_wind_forcing_from_file,
    run_model, zero_forcing,
)

print(backend_info())


def find_course_root():
    candidates = (
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "MT1562_python_lab_waves",
    )
    return next(path for path in candidates if (path / "notebooks").exists())


COURSE_ROOT = find_course_root()
DATA_DIR = COURSE_ROOT / "data"
ANIMATION_DIR = COURSE_ROOT / "animations"
ANIMATION_DIR.mkdir(exist_ok=True)
print("Course data:", DATA_DIR.resolve())


In [ ]:
def animate_with_overlay(
    out, grid, filename, *, title, draw_overlay=None, max_frames=42,
):
    frame_count = len(out["time"])
    frames = np.unique(
        np.linspace(0, frame_count - 1, min(max_frames, frame_count), dtype=int)
    )
    animation = animate_eta(
        out, grid, frames=frames, interval=100, repeat=True,
        title=title, contours=False, remove_mean=False,
        figsize=(9.0, 3.8),
    )
    if draw_overlay is not None:
        draw_overlay(animation.figure.axes[0])
    path = ANIMATION_DIR / filename
    animation.save(str(path), fps=10, dpi=90)
    print("Saved animation:", path.resolve())
    return animation


def cross_basin_pulse(
    grid, params, *, amplitude=0.10, radius=80e3, x0=350e3,
):
    eta_line = amplitude * np.exp(-((grid.x_c - x0) / radius) ** 2)
    eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)
    H_u = depth_on_u(grid, params.H)
    eta_u = amplitude * np.exp(-((grid.x_u - x0) / radius) ** 2)
    u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g/H_u)
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v


def rest_state(grid, params):
    return (
        np.zeros((grid.Ny, grid.Nx)),
        np.zeros((grid.Ny, grid.Nx + 1)),
        np.zeros((grid.Ny + 1, grid.Nx)),
    )


## 4. Bathymetry supplied as a file

The bathymetry loader accepts `.npy`, `.npz`, `.csv`, and `.txt` arrays.
Depth must be positive, finite, measured in metres, and have exact shape
`(Ny, Nx)`. The first index is south-to-north $y$ and the second is
west-to-east $x$. The loader does not interpolate, reproject, or create a
land mask.

Here the supplied two-dimensional shelf-and-ridge map becomes
`ModelParams.H`. Contour lines keep the bottom geometry visible while the
surface wave propagates over it.


In [ ]:
Nx, Ny = 160, 24
Lx, Ly = 2.4e6, 360e3
map_grid = make_grid(Nx, Ny, Lx, Ly)
H_map = load_bathymetry(DATA_DIR / "example_bathymetry.npz", map_grid)
print(
    "bathymetry shape:", H_map.shape,
    "depth range:", f"{H_map.min():.0f}–{H_map.max():.0f} m",
)

bathy_levels = np.linspace(H_map.min(), H_map.max(), 7)[1:-1]
fig, ax = plt.subplots(figsize=(9, 3.4))
field = ax.contourf(
    map_grid.x_c/1e3, map_grid.y_c/1e3, H_map,
    levels=18, cmap="Blues",
)
lines = ax.contour(
    map_grid.x_c/1e3, map_grid.y_c/1e3, H_map,
    levels=bathy_levels, colors="0.2", linewidths=0.7,
)
ax.clabel(lines, fmt="%.0f m", fontsize=7)
ax.set(xlabel="x [km]", ylabel="y [km]", title="Bathymetry input map")
fig.colorbar(field, ax=ax, label="depth [m]")
plt.show()


In [ ]:
bathy_params = ModelParams(
    H=H_map, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True,
)
bathy_dt = compute_dt_cfl(map_grid, bathy_params, cfl=0.42)
bathy_out = run_model(
    tmax=7*3600, dt=bathy_dt,
    grid=map_grid, params=bathy_params,
    forcing_fn=zero_forcing,
    ic_fn=lambda g, p: cross_basin_pulse(g, p),
    save_every=5, out_vars=("eta",),
)


def draw_bathymetry(ax):
    contours = ax.contour(
        map_grid.x_c, map_grid.y_c, H_map,
        levels=bathy_levels, colors="0.15", linewidths=0.65,
        alpha=0.75,
    )
    ax.clabel(contours, fmt="%.0f m", fontsize=6)


bathymetry_animation = animate_with_overlay(
    bathy_out, map_grid, "part_c_toolbox_bathymetry.gif",
    title="Wave over file-backed bathymetry",
    draw_overlay=draw_bathymetry,
)
bathymetry_animation


## 5. Wind forcing supplied as a file

The forcing file stores cell-centred stress components `tau_x` and
`tau_y` in $\mathrm{N\,m^{-2}}$. The loader reads the file once, places
the stresses on the staggered velocity grid, and returns a normal model
forcing function. A separate envelope can vary the amplitude in time.

The model is forced by stress rather than wind velocity. For display
only, the code estimates a wind speed from

$$
|\boldsymbol{\tau}|=\rho_{air} C_D U_{10}^2,
$$

using constant air density and drag coefficient. Wind-speed contours and
small direction arrows remain visible while the surface responds.


In [ ]:
with np.load(DATA_DIR / "example_wind_forcing.npz", allow_pickle=False) as data:
    tau_x_map = np.asarray(data["tau_x"])
    tau_y_map = np.asarray(data["tau_y"])

stress_magnitude = np.hypot(tau_x_map, tau_y_map)
rho_air, drag_coefficient = 1.225, 1.3e-3
wind_speed = np.sqrt(stress_magnitude / (rho_air * drag_coefficient))
direction_x = np.divide(
    tau_x_map, stress_magnitude,
    out=np.zeros_like(tau_x_map), where=stress_magnitude > 0,
)
direction_y = np.divide(
    tau_y_map, stress_magnitude,
    out=np.zeros_like(tau_y_map), where=stress_magnitude > 0,
)

wind_levels = np.linspace(1.0, max(2.0, float(wind_speed.max())), 6)
arrow_slice = (slice(1, None, 3), slice(4, None, 12))

fig, ax = plt.subplots(figsize=(9, 3.4))
speed_field = ax.contourf(
    map_grid.x_c/1e3, map_grid.y_c/1e3, wind_speed,
    levels=18, cmap="YlGnBu",
)
speed_lines = ax.contour(
    map_grid.x_c/1e3, map_grid.y_c/1e3, wind_speed,
    levels=wind_levels, colors="0.2", linewidths=0.65,
)
ax.clabel(speed_lines, fmt="%.1f m/s", fontsize=7)
ax.quiver(
    map_grid.x_c[arrow_slice[1]]/1e3,
    map_grid.y_c[arrow_slice[0]]/1e3,
    direction_x[arrow_slice], direction_y[arrow_slice],
    color="0.15", scale=28, width=0.0022, pivot="middle",
)
ax.set(xlabel="x [km]", ylabel="y [km]", title="Mapped wind pattern")
fig.colorbar(speed_field, ax=ax, label="estimated wind speed [m/s]")
plt.show()


In [ ]:
wind_map_forcing = make_wind_forcing_from_file(
    DATA_DIR / "example_wind_forcing.npz",
    map_grid,
    envelope=lambda t: 1.0,
)
wind_map_params = ModelParams(
    H=400.0, g=9.81, f0=0.0, beta=0.0,
    r=1/(2*86400), linear=True,
)
wind_map_dt = compute_dt_cfl(map_grid, wind_map_params, cfl=0.42)
wind_map_out = run_model(
    tmax=8*3600, dt=wind_map_dt,
    grid=map_grid, params=wind_map_params,
    forcing_fn=wind_map_forcing, ic_fn=rest_state,
    save_every=8, out_vars=("eta",),
)


def draw_wind_pattern(ax):
    contours = ax.contour(
        map_grid.x_c, map_grid.y_c, wind_speed,
        levels=wind_levels, colors="0.15", linewidths=0.65,
        alpha=0.75,
    )
    ax.clabel(contours, fmt="%.1f m/s", fontsize=6)
    ax.quiver(
        map_grid.x_c[arrow_slice[1]],
        map_grid.y_c[arrow_slice[0]],
        direction_x[arrow_slice], direction_y[arrow_slice],
        color="0.12", scale=28, width=0.0022, pivot="middle",
    )


wind_map_animation = animate_with_overlay(
    wind_map_out, map_grid, "part_c_toolbox_wind.gif",
    title="Surface response to mapped wind forcing",
    draw_overlay=draw_wind_pattern,
)
wind_map_animation


## 6. Turning the toolbox into a project

The demonstrations above are starting points, not prescribed projects.
A group might change one map amplitude, shelf depth, forcing duration,
domain dimension, initial-state scale, or Coriolis parameter. A compact
design looks like this:

| Element | Example |
|---|---|
| Question | How does eastward-wind duration affect eastern-wall setup? |
| Prediction | Longer forcing initially produces larger setup. |
| Baseline | 6-hour wind event; all other controls fixed. |
| Variations | Identical 3-hour and 9-hour events. |
| Diagnostic | Maximum mean $\eta$ in the four easternmost cells. |
| Limitation | A rigid wet wall is not a beach or inundation model. |

The report notebook already contains a reusable `run_case(...)`
function and the required report headings. Groups should copy only the
toolbox code relevant to their question, keep the report readable, and
submit the executed report notebook—not this demonstration notebook.
